In [38]:
import pandas as pd
from collections import defaultdict

file = '/mnt/c/Users/georg/Downloads/UpdatedTable.xlsx'
df = pd.read_excel(file, sheet_name='CleanTableFiltered')
df_filtered = df[df['Filtered'] == True]

In [39]:
uniprot_ids = df_filtered['UNIPROT ID'].tolist()

In [40]:
len(set(uniprot_ids))

550

In [41]:
# check if there are any spaces in the uniprot ids, if so count them and print them out
space_count = 0
for uniprot_id in uniprot_ids:
    if ' ' in uniprot_id:
        space_count += 1
        print(uniprot_id)

In [42]:
import requests
import time
import pandas as pd
from io import StringIO

# Example dummy list (replace with your uniprot_ids)
# uniprot_ids = ["P21802", "P12345", "Q9Y243"] 

# 1. Submit the batch job
submit_url = "https://rest.uniprot.org/idmapping/run"
payload = {
    "from": "UniProtKB_AC-ID",
    "to": "UniProtKB",
    "ids": ",".join(set(uniprot_ids))
}
response = requests.post(submit_url, data=payload)
job_id = response.json()["jobId"]
print(f"Job submitted. ID: {job_id}")

# 2. Check the status until it's finished
status_url = f"https://rest.uniprot.org/idmapping/status/{job_id}"
while True:
    status_response = requests.get(status_url).json()
    if status_response.get("jobStatus") in ["RUNNING", "NEW"]:
        print("Waiting for results...")
        time.sleep(3)  # Wait 3 seconds before checking again
    else:
        break

# 3. Download the results as a TSV
# We specifically request the accession (Entry) and organism_name (Organism)
results_url = f"https://rest.uniprot.org/idmapping/uniprotkb/results/stream/{job_id}"
params = {
    "format": "tsv",
    "fields": "accession,organism_name"
}
results_response = requests.get(results_url, params=params)

# 4. Parse the results and count
df_species = pd.read_csv(StringIO(results_response.text), sep="\t")

# The ID mapping tool adds a "From" column (your original ID) 
# and returns the requested fields ("Entry", "Organism")
if 'Organism' in df_species.columns:
    species_counts = df_species['Organism'].value_counts()
    print("\n--- Species Count ---")
    print(species_counts)
else:
    print("Organism column not found. Check if the IDs were valid.")

Job submitted. ID: 5x5kxF96iE
Waiting for results...
Waiting for results...

--- Species Count ---
Organism
Homo sapiens (Human)                                                     313
Mus musculus (Mouse)                                                     132
Rattus norvegicus (Rat)                                                   22
Arabidopsis thaliana (Mouse-ear cress)                                    12
Gallus gallus (Chicken)                                                   10
Xenopus laevis (African clawed frog)                                       9
Danio rerio (Zebrafish) (Brachydanio rerio)                                8
Drosophila melanogaster (Fruit fly)                                        7
Nicotiana tabacum (Common tobacco)                                         3
Triticum aestivum (Wheat)                                                  3
Zea mays (Maize)                                                           2
Sus scrofa (Pig)                             

In [43]:
# check if all rows with same uniprot ID have the same sequence, if not print out the uniprot ID and the number of different sequences
sequence_counts = df_filtered.groupby('UNIPROT ID')['UNIPROT SEQ'].nunique()
for uniprot_id, count in sequence_counts.items():
    if count > 1:
        print(f"{uniprot_id} has {count} different sequences.")

In [44]:
import requests
import time
import pandas as pd
from io import StringIO

# 1. Get all unique IDs from your dataframe
unique_ids = list(set(df_filtered['UNIPROT ID'].dropna().astype(str).tolist()))

# --- PHASE 1: Fetch from UniProtKB ---
def fetch_uniprotkb(ids, chunk_size=100):
    url = "https://rest.uniprot.org/uniprotkb/accessions"
    all_dataframes = []
    
    print(f"Phase 1: Fetching {len(ids)} IDs from UniProtKB...")
    for i in range(0, len(ids), chunk_size):
        chunk = ids[i:i + chunk_size]
        params = {
            "accessions": ",".join(chunk),
            "format": "tsv",
            "fields": "accession,sequence"
        }
        response = requests.get(url, params=params)
        if response.status_code == 200:
            df_chunk = pd.read_csv(StringIO(response.text), sep="\t")
            all_dataframes.append(df_chunk)
            
    return pd.concat(all_dataframes, ignore_index=True) if all_dataframes else pd.DataFrame()

df_kb = fetch_uniprotkb(unique_ids)

# --- PHASE 2: Fallback to UniParc for missing IDs ---
found_kb_ids = df_kb['Entry'].tolist() if not df_kb.empty else []
missing_ids = list(set(unique_ids) - set(found_kb_ids))
df_parc = pd.DataFrame()

if missing_ids:
    print(f"\nPhase 2: Found {len(missing_ids)} missing IDs. Searching UniParc Archive...")
    
    # Use ID Mapping to translate the obsolete UniProt IDs to UniParc
    submit_url = "https://rest.uniprot.org/idmapping/run"
    payload = {
        "from": "UniProtKB_AC-ID",
        "to": "UniParc",
        "ids": ",".join(missing_ids)
    }
    response = requests.post(submit_url, data=payload)
    job_id = response.json().get("jobId")
    
    # Wait for the job to finish
    status_url = f"https://rest.uniprot.org/idmapping/status/{job_id}"
    while True:
        status_res = requests.get(status_url).json()
        if status_res.get("jobStatus") in ["RUNNING", "NEW"]:
            time.sleep(3)
        else:
            break
            
    # Fetch UniParc results (we need 'sequence')
    results_url = f"https://rest.uniprot.org/idmapping/uniparc/results/stream/{job_id}"
    params = {
        "format": "tsv",
        "fields": "upi,sequence" 
    }
    res = requests.get(results_url, params=params)
    df_parc_raw = pd.read_csv(StringIO(res.text), sep="\t")
    
    # The ID mapping returns the queried ID in the 'From' column. 
    # We rename it to 'Entry' so it stacks perfectly with the UniProtKB data.
    if not df_parc_raw.empty and 'From' in df_parc_raw.columns:
        df_parc = df_parc_raw[['From', 'Sequence']].rename(columns={'From': 'Entry'})
        # Drop duplicates in case one obsolete ID maps to multiple UniParc entries
        df_parc = df_parc.drop_duplicates(subset=['Entry'])
        print(f"Recovered {len(df_parc)} sequences from UniParc!")

# --- PHASE 3: Combine and Validate ---
# Combine the valid UniProtKB matches and the recovered UniParc matches
df_official = pd.concat([df_kb, df_parc], ignore_index=True)

# Merge back with your original filtered dataframe
df_merged = pd.merge(
    df_filtered, 
    df_official, 
    left_on='UNIPROT ID', 
    right_on='Entry', 
    how='left'
)

# Clean whitespace and handle completely orphaned NaNs
df_merged['UNIPROT SEQ'] = df_merged['UNIPROT SEQ'].astype(str).str.strip().replace('nan', '')
df_merged['Sequence'] = df_merged['Sequence'].fillna('').astype(str).str.strip()

# Check 1: Sequence Match
df_merged['Sequence_Match'] = df_merged['UNIPROT SEQ'] == df_merged['Sequence']

# Check 2: Length Match (Your UNIPROT LENGTH vs the actual string length of the official sequence)
df_merged['Length_Match'] = df_merged['UNIPROT LENGTH'].astype(float) == df_merged['Sequence'].str.len().astype(float)

# Isolate the problem rows
mismatches = df_merged[(df_merged['Sequence_Match'] == False) | (df_merged['Length_Match'] == False)]

print(f"\n--- Final Validation Results ---")
print(f"Total rows in dataset: {len(df_merged)}")
print(f"Perfect rows: {len(df_merged) - len(mismatches)}")
print(f"Rows with issues: {len(mismatches)}")

if not mismatches.empty:
    print("\n--- Problematic Rows ---")
    columns_to_show = ['UNIPROT ID', 'Sequence_Match', 'Length_Match', 'UNIPROT LENGTH']
    mismatches_view = mismatches[columns_to_show].copy()
    mismatches_view['Official_Length'] = mismatches['Sequence'].str.len()
    print(mismatches_view.head(15))

Phase 1: Fetching 550 IDs from UniProtKB...

Phase 2: Found 3 missing IDs. Searching UniParc Archive...
Recovered 3 sequences from UniParc!

--- Final Validation Results ---
Total rows in dataset: 1053
Perfect rows: 1053
Rows with issues: 0


In [45]:
df_filtered

,PUBMED_ID,Gene,DomainCoordinates,DomainType,Species,UNIPROT ID,UNIPROT SEQ,UNIPROT LENGTH,Note,original_row,Filtered
0,17949687,AHRR,555-701,repression,Mouse,Q3U1U7-1,MMIPSGECTYAGRKRRKPIQKRRLTMGAEKSNPSKRHRDRLNTELD...,701,NaN,2,True
1,21047992,AHRR,217-402,repression,Chicken (Gallus gallus),E5L8C8,MIPPGECLYAGRKRRKPIQKQRPAAGNEKSNPSKRHRDRLNAELDH...,756,NaN,3,True
4,7488247,ARNT,704-789,activation,Human,P27540-1,MAATTANPEMTSDVPSLGPAIASGNSGPGIQGGGAIVQRAIKRRPG...,789,704-789 AD,6,True
5,7759522,AHR,490-718,activation,Not specified.,P30561,MSSGANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTEL...,848,AHR 490–718 AD,7,True
6,9111021,HIF1A,530-652,bifunctional,Human,Q16665-1,MEGAGGANDKKKISSERRKEKSRDAARSRRSKESEVFYELAHQLPL...,826,HIF1a 530-652 bifunctional and 652-826 AD,8,True
...,...,...,...,...,...,...,...,...,...,...,...
1572,20190276,GCM2,428-506,activation,Human,O75603,MPAAAVQEAVGVCSYGMQLSWDINDPQMPQELALFDQFREWPDGYV...,506,TAD: 428-506,1495,True
1573,20006729,LEF1,5-34,activation,Mouse (murine),P27782-1,MPQLSGGGGGGDPELCATDEMIPFKDEGDPQKEKIFAEISHPEEEG...,397,NaN,1496,True
1574,22718198,SP1,8-290,activation,Human,P08047-1,MSDQDHSMDEMTAVVKIEKGVGGNNGGNGNGGGAFSQARSSSTGSS...,785,NaN,1497,True
1576,15919722,TCF21,1-77,repression,Mouse,O35437,MSTGSLSDVEDLQEVEMLDCDSLKVDSNKEFGTSNESTEEGSNCEN...,179,RD: 1-77,1499,True


In [46]:
import requests
import time
import pandas as pd
from io import StringIO
import os

# Ensure the output directory exists
os.makedirs("./blast_table", exist_ok=True)

# --- 1. Fetch Official Species via ID Mapping ---
unique_ids = list(set(df_filtered['UNIPROT ID'].dropna().astype(str).tolist()))

print("Submitting ID Mapping job to fetch official taxonomy...")
submit_url = "https://rest.uniprot.org/idmapping/run"
payload = {
    "from": "UniProtKB_AC-ID",
    "to": "UniProtKB",
    "ids": ",".join(unique_ids)
}
response = requests.post(submit_url, data=payload)
job_id = response.json()["jobId"]

status_url = f"https://rest.uniprot.org/idmapping/status/{job_id}"
while True:
    status_response = requests.get(status_url).json()
    if status_response.get("jobStatus") in ["RUNNING", "NEW"]:
        time.sleep(3)
    else:
        break

print("Job complete. Downloading taxonomy data...")
results_url = f"https://rest.uniprot.org/idmapping/uniprotkb/results/stream/{job_id}"
params = {
    "format": "tsv",
    "fields": "accession,organism_name"
}
results_response = requests.get(results_url, params=params)
df_species = pd.read_csv(StringIO(results_response.text), sep="\t")

df_species_clean = df_species.drop_duplicates(subset=['From'])


# --- 2. Merge and Prepare ALL Rows (Human & Animal) ---
# Map the official 'Organism' string back to your dataframe using 'From'
df_filtered2 = pd.merge(
    df_filtered,
    df_species_clean[['From', 'Organism']], 
    left_on='UNIPROT ID',
    right_on='From',
    how='left'
)

# Keep EVERYTHING for the canonical BLAST
df_all_domains = df_filtered2.copy()
print(f"\nPreparing {len(df_all_domains)} total domains (Human & Animal) for canonical BLAST.")


# --- 3. Extract Sequences and Write FASTA ---
def extract_domain_sequence(row):
    seq = str(row['UNIPROT SEQ']).strip()
    coords = str(row['DomainCoordinates']).strip()
    
    # Catch weird formatting dashes
    coords = coords.replace('–', '-').replace('—', '-')
    
    if pd.isna(seq) or seq == '' or seq == 'nan' or pd.isna(coords):
        return ""
        
    try:
        # NEW: Handle the "last X amino acids" format (e.g., "-83:")
        if coords.startswith('-') and coords.endswith(':'):
            # Extract the number (ignoring the '-' at the start and ':' at the end)
            num = int(coords[1:-1]) 
            return seq[-num:] # Python slicing for the last N characters
            
        # STANDARD: Handle the normal "100-200" format
        if '-' in coords:
            parts = coords.split('-')
            if len(parts) >= 2:
                start_str = parts[0].strip()
                end_str = parts[1].strip()
                
                # Double-check that we actually have numbers to parse
                if not start_str or not end_str:
                    print(f"Warning: Row {row.name} has weird coordinates: '{coords}'")
                    return ""
                    
                # UniProt coordinates are 1-based, Python slicing is 0-based
                start = int(start_str) - 1 
                end = int(end_str)
                return seq[start:end]
                
        return ""
    except Exception as e:
        print(f"Error parsing row {row.name} with coords '{coords}': {e}")
        return ""

df_all_domains['Domain_Sequence'] = df_all_domains.apply(extract_domain_sequence, axis=1)

# Drop rows where coordinate extraction failed
df_queries = df_all_domains[df_all_domains['Domain_Sequence'] != ""]

# Save to a newly named FASTA file
fasta_filename = "./blast_table/all_queries.fasta"
with open(fasta_filename, "w") as f:
    for index, row in df_queries.iterrows():
        # Header format: >query_Index_Gene_UniprotID
        header = f">query_{index}_{row['Gene']}_{row['UNIPROT ID']}"
        sequence = row['Domain_Sequence']
        f.write(f"{header}\n{sequence}\n")

print(f"Successfully wrote {len(df_queries)} domain sequences to '{fasta_filename}' ready for BLAST.")

Submitting ID Mapping job to fetch official taxonomy...
Job complete. Downloading taxonomy data...

Preparing 1053 total domains (Human & Animal) for canonical BLAST.
Successfully wrote 1053 domain sequences to './blast_table/all_queries.fasta' ready for BLAST.


In [47]:
df_all_domains[df_all_domains['Domain_Sequence'] == ""]

,PUBMED_ID,Gene,DomainCoordinates,DomainType,Species,UNIPROT ID,UNIPROT SEQ,UNIPROT LENGTH,Note,original_row,Filtered,From,Organism,Domain_Sequence


In [33]:
import pandas as pd

print("--- Generating Custom CANONICAL TF BLAST Database ---")

csv_path = "./blast_table/TF_completeIDs_with_ensembl_canonical.csv" 
df_meta = pd.read_csv(csv_path)

# Clean IDs
df_meta['Translation ID Clean'] = df_meta['Translation ID'].astype(str).str.split('.').str[0]
df_meta['UniProt_Clean'] = df_meta['UniProt_ID'].astype(str).str.split('-').str[0]

# Filter strictly for Canonical rows
df_canon = df_meta[df_meta['Ensembl canonical'].astype(str).str.strip().str.lower() == 'canonical'].copy()

# Drop any rows that are missing sequences
valid_seqs = df_canon.dropna(subset=['Sequence aa', 'UniProt_Clean'])

# THE FIX: Drop duplicate UniProt IDs so BLAST doesn't crash!
valid_seqs = valid_seqs.drop_duplicates(subset=['UniProt_Clean'], keep='first')

fasta_out = "./blast_table/custom_canonical_tf_db.fasta"
with open(fasta_out, 'w') as f:
    for _, row in valid_seqs.iterrows():
        # Formatted as >sp|UniProt_ID|ENSP_ID
        header = f">sp|{row['UniProt_Clean']}|{row['Translation ID Clean']}"
        f.write(f"{header}\n{row['Sequence aa']}\n")

print(f"Success! Created custom FASTA with {len(valid_seqs)} strictly unique CANONICAL sequences at '{fasta_out}'")

--- Generating Custom CANONICAL TF BLAST Database ---
Success! Created custom FASTA with 1632 strictly unique CANONICAL sequences at './blast_table/custom_canonical_tf_db.fasta'


In [48]:
import subprocess
import os

# Define file names
query_file = "./blast_table/all_queries.fasta"
db_name = "./blast_table/custom_db/human_proteome_db"
output_file = "./blast_table/blast_results.tsv"

# We request format 6 (tabular) and specify exactly which columns we want.
# qseqid: Your query ID | sseqid: Human target ID | pident: % Identity 
# qstart/qend: Query coords | sstart/send: Target human coords
blast_cmd = [
    "blastp",
    "-query", query_file,
    "-db", db_name,
    "-out", output_file,
    "-evalue", "1000",          # Strict threshold to avoid random noise
    "-seg", "no",              # Don't mask low complexity regions (can hide real matches)
    "-comp_based_stats", "F",    # Disable composition-based stats for short domains
    "-max_target_seqs", "1",    # We only care about the absolute best human hit
    "-num_threads", "6",       # Use multiple threads for speed (adjust based on your CPU)
    "-outfmt", "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"
]

print("Running BLASTp... This should only take a few seconds for ~300 sequences.")
try:
    subprocess.run(blast_cmd, check=True)
    print(f"Success! Results saved to {output_file}")
except subprocess.CalledProcessError as e:
    print(f"BLAST failed: {e}")

Running BLASTp... This should only take a few seconds for ~300 sequences.


Success! Results saved to ./blast_table/blast_results.tsv


In [49]:
import os
import pandas as pd
import numpy as np

# --- 2. Parse BLAST Results & Apply Canonical Logic ---
print("\n--- Parsing BLAST Results and Applying >95% Logic ---")

# 1. Load your local Ensembl/Canonical mapping CSV
csv_path = "./blast_table/TF_completeIDs_with_ensembl_canonical.csv" 
df_meta = pd.read_csv(csv_path)

# Clean IDs for matching
df_meta['Translation ID Clean'] = df_meta['Translation ID'].astype(str).str.split('.').str[0]
df_meta['Gene_Upper'] = df_meta['Gene name (HGNC)'].astype(str).str.strip().str.upper()
df_meta['UniProt_Clean'] = df_meta['UniProt_ID'].astype(str).str.split('-').str[0] 

def build_canonical_dict(group_col):
    mapping = {}
    for key, group in df_meta.groupby(group_col):
        if key == 'NAN' or pd.isna(key) or key == '': continue
        if len(group) == 1:
            mapping[key] = group['Translation ID Clean'].iloc[0]
        else:
            canon_rows = group[group['Ensembl canonical'].astype(str).str.strip().str.lower() == 'canonical']
            if not canon_rows.empty:
                mapping[key] = canon_rows['Translation ID Clean'].iloc[0]
            else:
                mapping[key] = group['Translation ID Clean'].iloc[0]
    return mapping

gene_to_ensp = build_canonical_dict('Gene_Upper')
uniprot_to_ensp = build_canonical_dict('UniProt_Clean')

# 2. Parse BLAST results
blast_cols = [
    'qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen',
    'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore'
]
output_file = "./blast_table/blast_results.tsv"

if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
    df_blast = pd.read_csv(output_file, sep='\t', names=blast_cols)
    df_blast['original_index'] = df_blast['qseqid'].apply(lambda x: int(x.split('_')[1]))
    
    # Drop duplicated HSPs so rows don't balloon!
    df_blast = df_blast.drop_duplicates(subset=['original_index'], keep='first')
    
    def extract_uniprot_id(s):
        if pd.isna(s): return s
        parts = str(s).split('|')
        if len(parts) >= 2: return parts[1]
        return s
    
    df_blast['BLAST_UNIPROT_ID'] = df_blast['sseqid'].apply(extract_uniprot_id)
    df_blast['BLAST_DomainCoordinates'] = df_blast['sstart'].astype(str) + '-' + df_blast['send'].astype(str)
    df_blast = df_blast.rename(columns={'pident': 'BLAST_Identity_Pct', 'evalue': 'BLAST_E_Value'})
    
    merge_cols = ['original_index', 'BLAST_UNIPROT_ID', 'BLAST_DomainCoordinates', 'BLAST_Identity_Pct', 'BLAST_E_Value']
    df_blast_meta = df_blast[merge_cols].set_index('original_index')
    
    # 3. Join back to full dataframe
    df_final = df_all_domains.join(df_blast_meta)
    if 'Organism' in df_final.columns:
        df_final = df_final.rename(columns={'Organism': 'UNIPROT SPECIES'})

    # 4. Apply Custom Logic 
    def determine_final_humanization(row):
        is_human = 'Homo sapiens' in str(row.get('UNIPROT SPECIES', ''))
        ident = float(row['BLAST_Identity_Pct']) if pd.notna(row['BLAST_Identity_Pct']) else 0.0
        actual_identity = row['BLAST_Identity_Pct']
        
        if ident > 95.0:
            return row['BLAST_UNIPROT_ID'], row['BLAST_DomainCoordinates'], actual_identity, "Successful BLAST Hit (>95%)"
        else:
            if is_human:
                reason = f"Native Human Fallback (Weak Hit: {ident}%)" if ident > 0 else "Native Human Fallback (NO BLAST Hit)"
                return row['UNIPROT ID'], row['DomainCoordinates'], actual_identity, reason
            else:
                reason = f"Failed Non-Human (Weak Hit: {ident}%)" if ident > 0 else "Failed Non-Human (NO BLAST Hit)"
                return pd.NA, pd.NA, actual_identity, reason

    df_final[['Final_Human_UNIPROT_ID', 'Final_Human_DomainCoords', 'Final_Identity', 'Mapping_Status']] = df_final.apply(
        determine_final_humanization, axis=1, result_type='expand'
    )

    # 5. Map the Ensembl (ENSP) ID (FIXED to ignore Fallbacks!)
    def get_final_ensp(row):
        if pd.isna(row['Final_Human_UNIPROT_ID']): return pd.NA
        
        # FIX: Do not auto-assign canonical ENSP if it's a native human fallback! Let Step 5B handle it.
        if 'Native Human Fallback' in str(row.get('Mapping_Status', '')):
            return pd.NA
            
        gene = str(row['Gene']).strip().upper()
        if gene in gene_to_ensp: return gene_to_ensp[gene]
        uprot = str(row['Final_Human_UNIPROT_ID']).split('-')[0]
        if uprot in uniprot_to_ensp: return uniprot_to_ensp[uprot]
        return pd.NA

    df_final['Final_Human_ENSP_ID'] = df_final.apply(get_final_ensp, axis=1)


    # --- 5B. Isoform Rescue Mission (Exact Match & Length Match) ---
    print("\n--- Running Isoform Rescue for Missing ENSP IDs ---")
    
    df_meta['Protein length'] = pd.to_numeric(df_meta['Protein length'], errors='coerce')
    
    if 'UNIPROT LENGTH' in df_final.columns:
        df_final['UNIPROT LENGTH_Num'] = pd.to_numeric(df_final['UNIPROT LENGTH'], errors='coerce')
    else:
        df_final['UNIPROT LENGTH_Num'] = pd.NA

    def rescue_missing_ensp(row):
        if pd.notna(row['Final_Human_ENSP_ID']):
            return row['Final_Human_ENSP_ID'], row['Mapping_Status']
            
        if 'Native Human Fallback' not in str(row['Mapping_Status']):
            return pd.NA, row['Mapping_Status']
            
        if pd.isna(row['Final_Human_UNIPROT_ID']):
            return pd.NA, row['Mapping_Status']
            
        gene = str(row['Gene']).strip().upper()
        target_uprot = str(row['Final_Human_UNIPROT_ID']).strip()
        
        gene_rows = df_meta[df_meta['Gene_Upper'] == gene]
        if gene_rows.empty:
            return pd.NA, row['Mapping_Status']
            
        exact_match = gene_rows[gene_rows['UniProt_ID'].astype(str).str.strip() == target_uprot]
        if not exact_match.empty:
            return exact_match['Translation ID Clean'].iloc[0], str(row['Mapping_Status']) + " (Rescued via Exact Isoform)"
            
        target_length = row.get('UNIPROT LENGTH_Num')
        
        if pd.notna(target_length):
            length_match = gene_rows[gene_rows['Protein length'] == float(target_length)]
            if not length_match.empty:
                return length_match['Translation ID Clean'].iloc[0], str(row['Mapping_Status']) + " (Rescued via Length)"
                
        return pd.NA, row['Mapping_Status']

    df_final[['Final_Human_ENSP_ID', 'Mapping_Status']] = df_final.apply(
        rescue_missing_ensp, axis=1, result_type='expand'
    )
    
    rescued_count = df_final['Mapping_Status'].str.contains('Rescued', na=False).sum()
    print(f"Successfully rescued {rescued_count} missing Native Human Ensembl IDs using Isoform & Length matching!")


    # --- 5C. Extract the Final Human Domain Sequence ---
    print("\n--- Extracting Final Human Domain Sequences ---")

    # Build dictionary from the FULL CSV so it includes rescued non-canonical isoforms!
    ensp_to_seq = dict(zip(df_meta['Translation ID Clean'], df_meta['Sequence aa']))

    def get_final_sequence(row):
        ensp = str(row.get('Final_Human_ENSP_ID', ''))
        coords = str(row.get('Final_Human_DomainCoords', ''))
        
        if pd.isna(row.get('Final_Human_ENSP_ID')) or ensp == 'nan' or not coords or coords == 'nan':
            return ""
            
        full_seq = str(ensp_to_seq.get(ensp, ""))
        if not full_seq or full_seq == 'nan':
            return ""
            
        coords = coords.replace('–', '-').replace('—', '-')
        try:
            if coords.startswith('-') and coords.endswith(':'):
                num = int(coords[1:-1])
                return full_seq[-num:]
                
            if '-' in coords:
                start_str, end_str = coords.split('-')
                if start_str.strip() and end_str.strip():
                    start = int(start_str.strip()) - 1 
                    end = int(end_str.strip())
                    return full_seq[start:end]
        except Exception as e:
            pass
            
        return ""

    df_final['Final_Human_Domain_Sequence'] = df_final.apply(get_final_sequence, axis=1)
    
    seq_count = df_final[df_final['Final_Human_Domain_Sequence'] != ""].shape[0]
    print(f"Successfully extracted {seq_count} final human domain sequences!")


    print(f"\n--- Humanization & Canonical Mapping Complete ---")
    
    # 6. Save tracking lists for missing data
    missing_human_ensp = df_final[df_final['Final_Human_UNIPROT_ID'].notna() & df_final['Final_Human_ENSP_ID'].isna()]
    failed_non_humans = df_final[df_final['Mapping_Status'].str.contains('Failed Non-Human')]

    if len(missing_human_ensp) > 0:
        missing_file = "missing_human_ensembl_ids.csv"
        cols_to_save = ['Gene', 'UNIPROT SPECIES', 'UNIPROT ID', 'Final_Human_UNIPROT_ID', 'Mapping_Status']
        missing_human_ensp[cols_to_save].drop_duplicates().to_csv(missing_file, index=False)
        print(f"Saved {len(missing_human_ensp)} humanized rows missing Ensembl IDs to '{missing_file}'")
        
    if len(failed_non_humans) > 0:
        non_human_file = "failed_non_human_tfs.csv"
        cols_to_save = ['Gene', 'UNIPROT SPECIES', 'UNIPROT ID', 'Mapping_Status']
        failed_non_humans[cols_to_save].drop_duplicates().to_csv(non_human_file, index=False)
        print(f"Saved {len(failed_non_humans)} non-human TFs that failed to humanize to '{non_human_file}'")

    cols_to_show = ['Gene', 'Mapping_Status', 'Final_Human_ENSP_ID', 'Final_Human_Domain_Sequence']
    print("\nSample of finalized table:")
    print(df_final[cols_to_show].head(15))

else:
    print("\nNo BLAST results found or file is empty.")


--- Parsing BLAST Results and Applying >95% Logic ---

--- Running Isoform Rescue for Missing ENSP IDs ---
Successfully rescued 8 missing Native Human Ensembl IDs using Isoform & Length matching!

--- Extracting Final Human Domain Sequences ---
Successfully extracted 834 final human domain sequences!

--- Humanization & Canonical Mapping Complete ---
Saved 4 humanized rows missing Ensembl IDs to 'missing_human_ensembl_ids.csv'
Saved 215 non-human TFs that failed to humanize to 'failed_non_human_tfs.csv'

Sample of finalized table:
       Gene                                     Mapping_Status  \
0      AHRR               Failed Non-Human (Weak Hit: 48.701%)   
1      AHRR               Failed Non-Human (Weak Hit: 40.698%)   
2      ARNT                        Successful BLAST Hit (>95%)   
3       AHR               Failed Non-Human (Weak Hit: 57.456%)   
4     HIF1A                        Successful BLAST Hit (>95%)   
5     HIF1A                        Successful BLAST Hit (>95%)   


In [50]:
df_final

,PUBMED_ID,Gene,DomainCoordinates,DomainType,Species,UNIPROT ID,UNIPROT SEQ,UNIPROT LENGTH,Note,original_row,...,BLAST_DomainCoordinates,BLAST_Identity_Pct,BLAST_E_Value,Final_Human_UNIPROT_ID,Final_Human_DomainCoords,Final_Identity,Mapping_Status,Final_Human_ENSP_ID,UNIPROT LENGTH_Num,Final_Human_Domain_Sequence
0,17949687,AHRR,555-701,repression,Mouse,Q3U1U7-1,MMIPSGECTYAGRKRRKPIQKRRLTMGAEKSNPSKRHRDRLNTELD...,701,NaN,2,...,551-697,48.701,1.040000e-39,<NA>,<NA>,48.701,Failed Non-Human (Weak Hit: 48.701%),<NA>,701,
1,21047992,AHRR,217-402,repression,Chicken (Gallus gallus),E5L8C8,MIPPGECLYAGRKRRKPIQKQRPAAGNEKSNPSKRHRDRLNAELDH...,756,NaN,3,...,220-390,40.698,1.690000e-42,<NA>,<NA>,40.698,Failed Non-Human (Weak Hit: 40.698%),<NA>,756,
2,7488247,ARNT,704-789,activation,Human,P27540-1,MAATTANPEMTSDVPSLGPAIASGNSGPGIQGGGAIVQRAIKRRPG...,789,704-789 AD,6,...,704-789,100.000,4.280000e-58,P27540,704-789,100.000,Successful BLAST Hit (>95%),ENSP00000351407,789,FAPETGQTAGQFQTRTAEGVGVWPQWQGQQPHHRSSSSEQHVQQPP...
3,7759522,AHR,490-718,activation,Not specified.,P30561,MSSGANITYASRKRRKPVQKTVKPIPAEGIKSNPSKRHRDRLNTEL...,848,AHR 490–718 AD,7,...,501-716,57.456,4.670000e-79,<NA>,<NA>,57.456,Failed Non-Human (Weak Hit: 57.456%),<NA>,848,
4,9111021,HIF1A,530-652,bifunctional,Human,Q16665-1,MEGAGGANDKKKISSERRKEKSRDAARSRRSKESEVFYELAHQLPL...,826,HIF1a 530-652 bifunctional and 652-826 AD,8,...,530-652,100.000,4.110000e-78,Q16665,530-652,100.000,Successful BLAST Hit (>95%),ENSP00000338018,826,EFKLELVEKLFAEDTEAKNPFSTQDTDLDLEMLAPYIPMDDDFQLR...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048,20190276,GCM2,428-506,activation,Human,O75603,MPAAAVQEAVGVCSYGMQLSWDINDPQMPQELALFDQFREWPDGYV...,506,TAD: 428-506,1495,...,428-506,100.000,4.030000e-54,O75603,428-506,100.000,Successful BLAST Hit (>95%),ENSP00000368805,506,EPWGPPVTVTRAASPSGPPPMKIAGDCRAIRPTVAIPHEPVSSRTD...
1049,20006729,LEF1,5-34,activation,Mouse (murine),P27782-1,MPQLSGGGGGGDPELCATDEMIPFKDEGDPQKEKIFAEISHPEEEG...,397,NaN,1496,...,8-36,100.000,4.200000e-17,Q9UJU2,8-36,100.000,Successful BLAST Hit (>95%),ENSP00000265165,397,GGGGGGDPELCATDEMIPFKDEGDPQKEK
1050,22718198,SP1,8-290,activation,Human,P08047-1,MSDQDHSMDEMTAVVKIEKGVGGNNGGNGNGGGAFSQARSSSTGSS...,785,NaN,1497,...,8-290,100.000,0.000000e+00,P08047,8-290,100.000,Successful BLAST Hit (>95%),ENSP00000329357,785,MDEMTAVVKIEKGVGGNNGGNGNGGGAFSQARSSSTGSSSSTGGGG...
1051,15919722,TCF21,1-77,repression,Mouse,O35437,MSTGSLSDVEDLQEVEMLDCDSLKVDSNKEFGTSNESTEEGSNCEN...,179,RD: 1-77,1499,...,1-77,93.506,4.310000e-48,<NA>,<NA>,93.506,Failed Non-Human (Weak Hit: 93.506%),<NA>,179,


In [51]:
df_final.to_excel("/mnt/c/Users/georg/Downloads/HumanizedTable.xlsx", index=False)

In [148]:
# --- 3. Troubleshooting Failed Rows ---

print("\n--- Troubleshooting Failed & Dropped Rows ---")

# 1. Catch rows that failed sequence extraction (never made it to BLAST)
df_extraction_failures = df_non_human[df_non_human['Domain_Sequence'] == ""]
print(f"1. Sequence Extraction Failures: {len(df_extraction_failures)}")
if not df_extraction_failures.empty:
    print("These rows had missing sequences or invalid DomainCoordinates:")
    cols = ['Gene', 'UNIPROT ID', 'DomainCoordinates', 'UNIPROT SEQ']
    cols = [c for c in cols if c in df_extraction_failures.columns]
    print(df_extraction_failures[cols].head(5))


# 2. Catch rows that were queried but failed BLAST (or failed the >50% identity filter)
if 'df_metadata' in locals() and not df_queries.empty:
    queried_indices = df_queries.index
    successful_indices = df_metadata.index
    
    # Find indices that are in the query list but missing from the successful results
    failed_indices = queried_indices.difference(successful_indices)
    df_failed_blast = df_queries.loc[failed_indices]
    
    print(f"\n2. BLAST or Filter Failures: {len(df_failed_blast)}")
    if not df_failed_blast.empty:
        print("These rows went to BLAST but returned no valid human orthologs (>50% identity):")
        
        # We can also check if they had a hit < 50% by looking at the raw df_blast before filtering
        if 'df_blast' in locals():
            # Re-read raw blast if we overwrote it, or just do a quick check against the raw results
            raw_blast_df = pd.read_csv(output_file, sep='\t', names=blast_cols)
            raw_blast_df['original_index'] = raw_blast_df['qseqid'].apply(lambda x: int(x.split('_')[1]))
            
            # Map back the raw max identity to see how close they were
            max_identities = raw_blast_df.groupby('original_index')['pident'].max()
            df_failed_blast = df_failed_blast.copy()
            df_failed_blast['Best_Failed_Identity'] = df_failed_blast.index.map(max_identities)
            
        cols = ['Gene', 'UNIPROT ID', 'Domain_Sequence', 'Best_Failed_Identity']
        cols = [c for c in cols if c in df_failed_blast.columns]
        print(df_failed_blast[cols].head(15))


--- Troubleshooting Failed & Dropped Rows ---
1. Sequence Extraction Failures: 0

2. BLAST or Filter Failures: 75
These rows went to BLAST but returned no valid human orthologs (>50% identity):
      Gene UNIPROT ID                                    Domain_Sequence  \
0     AHRR   Q3U1U7-1  ASTTSCLWLGTSDMARGHLVGFPARMHLKTEPDYRQQACTPHLGHG...   
1     AHRR     E5L8C8  FICRVRCLLDSTSGFLTMQFQGKLKFLFGQRKKSSSGAVLPPQLSL...   
34     ERF   Q9SSA8-1  FPEENMKANSQKRSVKANLQKPVAKPNPNPSPALVQNSNISFENMC...   
35     ERF   Q9SSA8-1  EEKHQVSNNNNNQFGMTNSVDAGCNGYQYFSSDQGSNSFDCSEFGW...   
36     ERF     D9ICY5  PNEDDEYSIQARNPIPPLPFAPQHPPLYQQQYRCDLNNAPKNLNFE...   
50   FOXO1     Q9R1E0                           AAAGPLAGQPRKTSSSRRNAWGNL   
71    IRF5     F1QNG2  ETMDLNLHAVSQSLKTYSQPNIQTTSSNYFETTYSDDPCMQNNIPA...   
73    IRF7     P70434  VGPATENREEVSLSNALPTQGVSPGSFLARENAGLQTPSPLLSSDA...   
74    IRF7     P70434                GVSSLDSSSLGLCLSSTNSLYEDIEHFLMDLGQWP   
79    LHX3     Q25132  QLPSGPTSPISAPVTTGQKKRS

In [158]:
df_failed_blast['UNIPROT ID'].to_list()[2]

'Q9SSA8-1'

In [212]:
# --- 3. Fast Ensembl Mapping & Isolate Failures ---

print("\n--- Mapping UniProt IDs to Ensembl Protein (ENSP) IDs ---")

# 1. Isolate the rows that successfully got a human mapping
df_mapped = df_final[df_final['Human_UNIPROT_ID'].notna()].copy()

# Get unique human UniProt IDs
unique_human_ids = list(set(df_mapped['Human_UNIPROT_ID'].dropna().astype(str).tolist()))

# --- Step 3A: Fast API Fetch (For Active/Canonical IDs) ---
url = "https://rest.uniprot.org/uniprotkb/accessions"
ensp_mappings = []

for i in range(0, len(unique_human_ids), 100):
    chunk = unique_human_ids[i:i + 100]
    params = {
        "accessions": ",".join(chunk),
        "format": "tsv",
        "fields": "accession,xref_ensembl_full"
    }
    res = requests.get(url, params=params)
    if res.status_code == 200:
        df_chunk = pd.read_csv(StringIO(res.text), sep='\t')
        ensp_mappings.append(df_chunk)

df_ensp_clean = pd.DataFrame()

if ensp_mappings:
    df_ensp_raw = pd.concat(ensp_mappings, ignore_index=True)
    ensembl_col = next((col for col in df_ensp_raw.columns if 'Ensembl' in col), None)
    
    if ensembl_col:
        df_ensp_raw['ENSP_ID'] = df_ensp_raw[ensembl_col].astype(str).str.extract(r'(ENSP\d+)')
        df_ensp_clean = df_ensp_raw[['Entry', 'ENSP_ID']].dropna().rename(columns={'Entry': 'Human_UNIPROT_ID'})
        df_ensp_clean = df_ensp_clean.drop_duplicates(subset=['Human_UNIPROT_ID'])

# Merge the fast results
df_mapped_final = pd.DataFrame()
if not df_ensp_clean.empty:
    df_mapped_final = pd.merge(df_mapped, df_ensp_clean, on='Human_UNIPROT_ID', how='inner')
    print(f"Fast mapping caught {len(df_mapped_final)} rows.")

# --- Step 3B: Isolate and Save Failures for Manual Review ---
successful_ids = df_ensp_clean['Human_UNIPROT_ID'].tolist() if not df_ensp_clean.empty else []
missing_ids = [uid for uid in unique_human_ids if uid not in successful_ids]

if missing_ids:
    print(f"\nFound {len(missing_ids)} IDs that failed fast mapping (likely obsolete).")
    
    # Isolate the exact rows that failed so you know what genes they belong to
    df_failures = df_mapped[df_mapped['Human_UNIPROT_ID'].isin(missing_ids)].copy()
    
    print("\n--- IDs Requiring Manual Inspection ---")
    cols_to_show = ['Gene', 'UNIPROT ID', 'Human_UNIPROT_ID']
    print(df_failures[cols_to_show].drop_duplicates(subset=['Human_UNIPROT_ID']))


# --- Step 3C: Generate BED File (With Manual Review Rows at the Bottom) ---
print("\n--- Generating BED File for protein2genomic ---")

import os

bed_dfs_to_concat = []

# 1. Process the 965 Successful Hits
if not df_mapped_final.empty and 'ENSP_ID' in df_mapped_final.columns:
    df_success = df_mapped_final[df_mapped_final['Human_DomainCoordinates'].astype(str).str.contains('-')].copy()
    coords = df_success['Human_DomainCoordinates'].astype(str).str.split('-', expand=True)
    
    bed_success = pd.DataFrame()
    bed_success['protein_id'] = df_success['ENSP_ID']
    bed_success['aa_start'] = coords[0].str.strip()
    bed_success['aa_end'] = coords[1].str.strip()
    
    pmid_col = df_success.get('PUBMED_ID', pd.Series(['NoPMID'] * len(df_success)))
    bed_success['domain_id'] = df_success['Gene'].astype(str) + "_" + df_success['UNIPROT ID'].astype(str) + "_PMID:" + pmid_col.fillna('NoPMID').astype(str)
    
    bed_dfs_to_concat.append(bed_success)

# 2. Process the 12 Failures (Append at the end for manual review)
if 'df_failures' in locals() and not df_failures.empty:
    # Use the original non-human 'DomainCoordinates' as placeholders so you have a reference
    df_fails = df_failures[df_failures['DomainCoordinates'].astype(str).str.contains('-')].copy()
    fail_coords = df_fails['DomainCoordinates'].astype(str).str.split('-', expand=True)
    
    bed_fails = pd.DataFrame()
    # Add an obvious 'FIXME' flag so you can easily Ctrl+F them tomorrow
    bed_fails['protein_id'] = "FIXME_ENSP_FOR_" + df_fails['UNIPROT ID'].astype(str)
    bed_fails['aa_start'] = fail_coords[0].str.strip() 
    bed_fails['aa_end'] = fail_coords[1].str.strip()
    
    pmid_col_fails = df_fails.get('PUBMED_ID', pd.Series(['NoPMID'] * len(df_fails)))
    bed_fails['domain_id'] = df_fails['Gene'].astype(str) + "_" + df_fails['UNIPROT ID'].astype(str) + "_PMID:" + pmid_col_fails.fillna('NoPMID').astype(str)
    
    bed_dfs_to_concat.append(bed_fails)

# 3. Combine and Save
if bed_dfs_to_concat:
    final_bed_df = pd.concat(bed_dfs_to_concat, ignore_index=True)
    
    # Ensure the directory exists
    bed_filename = "./protein2genomic/domains.bed"
    os.makedirs(os.path.dirname(bed_filename), exist_ok=True)
    
    # Write the file
    final_bed_df.to_csv(bed_filename, sep='\t', header=False, index=False)
    
    print(f"Successfully wrote BED file to '{bed_filename}'")
    print(f"Total rows in BED: {len(final_bed_df)} (Mapped + Manual Review)")
    print("\nSample of the BOTTOM of domains.bed (Your manual review rows):")
    print(final_bed_df.tail(15))
else:
    print("No valid domains found. BED file not created.")


--- Mapping UniProt IDs to Ensembl Protein (ENSP) IDs ---
Fast mapping caught 965 rows.

Found 12 IDs that failed fast mapping (likely obsolete).

--- IDs Requiring Manual Inspection ---
       Gene UNIPROT ID Human_UNIPROT_ID
156  POU3F4     Q812B1           P49335
178    RORA   P35398-4         P35398-4
275     WT1   P19544-1         P19544-1
287    ZHX1   Q9UKY1-1         Q9UKY1-1
311   NANOG     Q80Z64           Q8N7R0
359   NFAT5   O94916-2         O94916-2
380  SREBF1   P36956-5         P36956-5
381  SREBF1   P36956-6         P36956-6
516    KLF4   O43474-3         O43474-3
650  PRDM16   Q9HAZ2-4         Q9HAZ2-4
682   CREB3   O43889-3         O43889-3
758    MZF1   P28698-2         P28698-2

--- Generating BED File for protein2genomic ---
Successfully wrote BED file to './protein2genomic/domains.bed'
Total rows in BED: 968 (Mapped + Manual Review)

Sample of the BOTTOM of domains.bed (Your manual review rows):
                  protein_id aa_start aa_end                      do

In [207]:
len(df_final[df_final['Human_UNIPROT_ID'].notna()])

982

In [206]:
len(df_final['Human_UNIPROT_ID'].dropna().astype(str))

982

### Tfregdb1

In [53]:
import pandas as pd
from collections import defaultdict

file = '/mnt/c/Users/georg/Downloads/UpdatedTable.xlsx'
df = pd.read_excel(file, sheet_name='TFRegDB1')

fasta_out = "./blast_table/tfregdb1_queries.fasta"
with open(fasta_out, 'w') as f:
    for index, row in df.iterrows():
        # Header: >query_Index_TFname_UniprotID
        header = f">query_{index}_{row['TF name']}_{row['Uniprot ID']}"
        f.write(f"{header}\n{row['Sequence']}\n")

print(f"Generated {len(df)} queries at {fasta_out}")

Generated 924 queries at ./blast_table/tfregdb1_queries.fasta


In [54]:
import subprocess
import os

# Define file names
query_file = "./blast_table/tfregdb1_queries.fasta"
db_name = "./blast_table/custom_db/human_proteome_db"
output_file = "./blast_table/blast_results_tfregdb1.tsv"

# We request format 6 (tabular) and specify exactly which columns we want.
# qseqid: Your query ID | sseqid: Human target ID | pident: % Identity 
# qstart/qend: Query coords | sstart/send: Target human coords
blast_cmd = [
    "blastp",
    "-query", query_file,
    "-db", db_name,
    "-out", output_file,
    "-evalue", "1000",          # Strict threshold to avoid random noise
    "-seg", "no",              # Don't mask low complexity regions (can hide real matches)
    "-comp_based_stats", "F",    # Disable composition-based stats for short domains
    "-max_target_seqs", "1",    # We only care about the absolute best human hit
    "-num_threads", "6",       # Use multiple threads for speed (adjust based on your CPU)
    "-outfmt", "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"
]

print("Running BLASTp... This should only take a few seconds for ~300 sequences.")
try:
    subprocess.run(blast_cmd, check=True)
    print(f"Success! Results saved to {output_file}")
except subprocess.CalledProcessError as e:
    print(f"BLAST failed: {e}")

Running BLASTp... This should only take a few seconds for ~300 sequences.


Success! Results saved to ./blast_table/blast_results_tfregdb1.tsv


In [55]:
import os
import pandas as pd
import numpy as np

print("\n--- Processing Effector Domains & Applying >95% Logic ---")

# --- 1. Load Data ---
# Load your local Canonical mapping CSV
df_meta = pd.read_csv("./blast_table/TF_completeIDs_with_ensembl_canonical.csv")
df_meta['Translation ID Clean'] = df_meta['Translation ID'].astype(str).str.split('.').str[0]
df_meta['Gene_Upper'] = df_meta['Gene name (HGNC)'].astype(str).str.strip().str.upper()
df_meta['UniProt_Clean'] = df_meta['UniProt_ID'].astype(str).str.split('-').str[0] 

# Load the NEW Effector table
df_eff = pd.read_excel("/mnt/c/Users/georg/Downloads/UpdatedTable.xlsx", sheet_name='TFRegDB1')

def build_canonical_dict(group_col):
    mapping = {}
    for key, group in df_meta.groupby(group_col):
        if key == 'NAN' or pd.isna(key) or key == '': continue
        if len(group) == 1:
            mapping[key] = group['Translation ID Clean'].iloc[0]
        else:
            canon_rows = group[group['Ensembl canonical'].astype(str).str.strip().str.lower() == 'canonical']
            if not canon_rows.empty:
                mapping[key] = canon_rows['Translation ID Clean'].iloc[0]
            else:
                mapping[key] = group['Translation ID Clean'].iloc[0]
    return mapping

gene_to_ensp = build_canonical_dict('Gene_Upper')
uniprot_to_ensp = build_canonical_dict('UniProt_Clean')
ensp_to_seq = dict(zip(df_meta['Translation ID Clean'], df_meta['Sequence aa']))

# --- 2. Parse BLAST Results ---
blast_cols = [
    'qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen',
    'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore'
]
output_file = "./blast_table/blast_results_tfregdb1.tsv"

if os.path.exists(output_file) and os.path.getsize(output_file) > 0:
    df_blast = pd.read_csv(output_file, sep='\t', names=blast_cols)
    df_blast['original_index'] = df_blast['qseqid'].apply(lambda x: int(x.split('_')[1]))
    df_blast = df_blast.drop_duplicates(subset=['original_index'], keep='first')
    
    def extract_uniprot_id(s):
        if pd.isna(s): return s
        parts = str(s).split('|')
        if len(parts) >= 2: return parts[1]
        return s
    
    df_blast['BLAST_UNIPROT_ID'] = df_blast['sseqid'].apply(extract_uniprot_id)
    df_blast['BLAST_DomainCoordinates'] = df_blast['sstart'].astype(str) + '-' + df_blast['send'].astype(str)
    df_blast = df_blast.rename(columns={'pident': 'BLAST_Identity_Pct'})
    
    merge_cols = ['original_index', 'BLAST_UNIPROT_ID', 'BLAST_DomainCoordinates', 'BLAST_Identity_Pct']
    df_blast_meta = df_blast[merge_cols].set_index('original_index')
    
    # Join back to effector dataframe
    df_final = df_eff.join(df_blast_meta)

    # --- 3. Apply Custom Logic ---
    def determine_final_humanization(row):
        ident = float(row['BLAST_Identity_Pct']) if pd.notna(row['BLAST_Identity_Pct']) else 0.0
        actual_identity = row['BLAST_Identity_Pct']
        
        # Condition A: >95% BLAST Hit
        if ident > 95.0:
            return row['BLAST_UNIPROT_ID'], row['BLAST_DomainCoordinates'], actual_identity, "Successful BLAST Hit (>95%)"
        # Condition B: Native Human Fallback (Since all are human)
        else:
            reason = f"Native Human Fallback (Weak Hit: {ident}%)" if ident > 0 else "Native Human Fallback (NO BLAST Hit)"
            # Use original table columns: Uniprot ID & Coordinates
            return row['Uniprot ID'], row['Coordinates'], actual_identity, reason

    df_final[['Final_Human_UNIPROT_ID', 'Final_Human_DomainCoords', 'Final_Identity', 'Mapping_Status']] = df_final.apply(
        determine_final_humanization, axis=1, result_type='expand'
    )

    # --- 4. Map the Ensembl (ENSP) ID ---
    def get_final_ensp(row):
        # 1. If it's a fallback, trust the ENSEMBL protein ID from the new table!
        if 'Native Human Fallback' in str(row.get('Mapping_Status', '')):
            existing_ensp = str(row.get('ENSEMBL protein ID', '')).split('.')[0]
            if existing_ensp and existing_ensp != 'nan':
                return existing_ensp
                
        # 2. If it's a BLAST hit, map it to Canonical ENSP
        if pd.isna(row['Final_Human_UNIPROT_ID']): return pd.NA
        gene = str(row['TF name']).strip().upper()
        if gene in gene_to_ensp: return gene_to_ensp[gene]
        uprot = str(row['Final_Human_UNIPROT_ID']).split('-')[0]
        if uprot in uniprot_to_ensp: return uniprot_to_ensp[uprot]
        
        return pd.NA

    df_final['Final_Human_ENSP_ID'] = df_final.apply(get_final_ensp, axis=1)

    # --- 5. Extract Final Domain Sequence ---
    print("\n--- Extracting Final Human Domain Sequences ---")
    def get_final_sequence(row):
        ensp = str(row.get('Final_Human_ENSP_ID', ''))
        coords = str(row.get('Final_Human_DomainCoords', ''))
        
        if pd.isna(row.get('Final_Human_ENSP_ID')) or ensp == 'nan' or not coords or coords == 'nan':
            return ""
            
        full_seq = str(ensp_to_seq.get(ensp, ""))
        if not full_seq or full_seq == 'nan':
            return ""
            
        coords = coords.replace('–', '-').replace('—', '-')
        try:
            if coords.startswith('-') and coords.endswith(':'):
                num = int(coords[1:-1])
                return full_seq[-num:]
                
            if '-' in coords:
                start_str, end_str = coords.split('-')
                if start_str.strip() and end_str.strip():
                    start = int(start_str.strip()) - 1 
                    end = int(end_str.strip())
                    return full_seq[start:end]
        except Exception as e:
            pass
            
        return ""

    df_final['Final_Human_Domain_Sequence'] = df_final.apply(get_final_sequence, axis=1)
    
    # --- 6. Quick Output ---
    seq_count = df_final[df_final['Final_Human_Domain_Sequence'] != ""].shape[0]
    print(f"Successfully extracted {seq_count} final domain sequences!")
    
    cols_to_show = ['TF name', 'Mapping_Status', 'Final_Human_ENSP_ID', 'Final_Human_Domain_Sequence']
    print("\nSample of finalized table:")
    print(df_final[cols_to_show].head(15))
    
    # Save the final file
    # df_final.to_csv("tfregdb1_processed.csv", index=False)
    # print("\nSaved fully processed table to 'tfregdb1_processed.csv'!")

else:
    print("\nNo BLAST results found or file is empty.")


--- Processing Effector Domains & Applying >95% Logic ---

--- Extracting Final Human Domain Sequences ---
Successfully extracted 924 final domain sequences!

Sample of finalized table:
   TF name               Mapping_Status Final_Human_ENSP_ID  \
0    AEBP2  Successful BLAST Hit (>95%)     ENSP00000266508   
1   AHCTF1  Successful BLAST Hit (>95%)     ENSP00000497061   
2   AHCTF1  Successful BLAST Hit (>95%)     ENSP00000497061   
3   AHCTF1  Successful BLAST Hit (>95%)     ENSP00000497061   
4      AHR  Successful BLAST Hit (>95%)     ENSP00000242057   
5     AIRE  Successful BLAST Hit (>95%)     ENSP00000291582   
6   AKAP8L  Successful BLAST Hit (>95%)     ENSP00000380557   
7     ALX1  Successful BLAST Hit (>95%)     ENSP00000315417   
8     ALX3  Successful BLAST Hit (>95%)     ENSP00000497310   
9     ALX3  Successful BLAST Hit (>95%)     ENSP00000497310   
10      AR  Successful BLAST Hit (>95%)     ENSP00000363822   
11      AR  Successful BLAST Hit (>95%)     ENSP000003638

In [57]:
df_final.to_excel("/mnt/c/Users/georg/Downloads/TFRegDB1_Humanized.xlsx", index=False)

In [60]:
# wanna see rows where Coordinates are the same as Final_Human_DomainCoords and had 100% identity

df_check = df_final[(df_final['Coordinates'] != df_final['Final_Human_DomainCoords']) & (df_final['Final_Identity'] == 100.0)]
df_check

,Effector domain ID,TF name,TF Family,Domain type,Uniprot ID,Coordinates,Sequence,ENSEMBL gene ID,ENSEMBL protein ID,Assay,...,"Confidence (H, M or L)",BLAST_UNIPROT_ID,BLAST_DomainCoordinates,BLAST_Identity_Pct,Final_Human_UNIPROT_ID,Final_Human_DomainCoords,Final_Identity,Mapping_Status,Final_Human_ENSP_ID,Final_Human_Domain_Sequence
65,Effector 0066,DACH1,Unknown,RD,Q9UI36,618-758,IETLLTNIQGLLKVAIDNARAQEKQVQLEKTELKMDFLRERELRET...,ENSG00000276644,ENSP00000482797,"Luciferase assay, and binding to TCERG1",...,M,Q9UI36,566-706,100.0,Q9UI36,566-706,100.0,Successful BLAST Hit (>95%),ENSP00000482245,IETLLTNIQGLLKVAIDNARAQEKQVQLEKTELKMDFLRERELRET...
99,Effector 0100,ELF5,Ets,AD,Q9UKW6,35-139,DLFSNEEYYPAFEHQTACDSYWTSVHPEYWTKRHVWEWLQFCCDQY...,ENSG00000135374,ENSP00000311010,Gal4 DB fusion and luciferase assay,...,H,Q9UKW6,25-129,100.0,Q9UKW6,25-129,100.0,Successful BLAST Hit (>95%),ENSP00000257832,DLFSNEEYYPAFEHQTACDSYWTSVHPEYWTKRHVWEWLQFCCDQY...
110,Effector 0111,ERG,Ets,AD,P11308,125-209,MTTNERRVIVPADPTLWSTDHVRQWLEWAVKEYGLPDVNILLFQNI...,ENSG00000157554,ENSP00000414150,Not specified,...,M,P11308,118-202,100.0,P11308,118-202,100.0,Successful BLAST Hit (>95%),ENSP00000288319,MTTNERRVIVPADPTLWSTDHVRQWLEWAVKEYGLPDVNILLFQNI...
111,Effector 0112,ERG,Ets,AD,P11308,440-486,PHPPALPVTSSSFFAAPNPYWNSPTGGIYPNTRLPTSHMPSHLGTYY,ENSG00000157554,ENSP00000414150,Luciferase assay,...,M,P11308,433-479,100.0,P11308,433-479,100.0,Successful BLAST Hit (>95%),ENSP00000288319,PHPPALPVTSSSFFAAPNPYWNSPTGGIYPNTRLPTSHMPSHLGTYY
118,Effector 0119,ESRRB,Nuclear receptor,AD,O95718,1-99,MSSDDRHLGSSCGSFIKTEPSSPSSGIDALSHHSPSGSSDASGGFG...,ENSG00000119715,ENSP00000370270,Not specified,...,M,A0A2R8Y491,22-120,100.0,A0A2R8Y491,22-120,100.0,Successful BLAST Hit (>95%),ENSP00000493776,MSSDDRHLGSSCGSFIKTEPSSPSSGIDALSHHSPSGSSDASGGFG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
857,Effector 0858,ZNF250,C2H2 ZF,RD,P15622,52-163,METYGNVVSLGLPGSKPDIISQLERGEDPWVLDRKGAKKSQGLWSD...,ENSG00000196150,ENSP00000292579,dCas9 fusion and reporter gene expression,...,H,P15622,47-158,100.0,P15622,47-158,100.0,Successful BLAST Hit (>95%),ENSP00000393442,METYGNVVSLGLPGSKPDIISQLERGEDPWVLDRKGAKKSQGLWSD...
872,Effector 0873,ZNF34,C2H2 ZF,RD,Q8IZ26,37-149,FEDVAVYLSREEWGRLGPAQRGLYRDVMLETYGNLVSLGVGPAGPK...,ENSG00000196378,ENSP00000341528,dCas9 fusion and reporter gene expression,...,M,A0A0C4DG42,16-128,100.0,A0A0C4DG42,16-128,100.0,Successful BLAST Hit (>95%),ENSP00000396894,FEDVAVYLSREEWGRLGPAQRGLYRDVMLETYGNLVSLGVGPAGPK...
902,Effector 0903,ZNF641,C2H2 ZF,RD,Q96N77,1-132,MQAEDRSQFGSAAEMLSEQTAALGTGWESMNVQLDGAEPQVERGSQ...,ENSG00000167528,ENSP00000437832,Gal4 DB fusion and luciferase assay,...,H,Q96N77,1-118,100.0,Q96N77,1-118,100.0,Successful BLAST Hit (>95%),ENSP00000449974,MLSEQTAALGTGWESMNVQLDGAEPQVERGSQEERPWRTVPGPLEH...
903,Effector 0904,ZNF641,C2H2 ZF,AD,Q96N77,171-262,PDPQDLEERDILRVTYTGDGSEHEGDTPELEAEPPRMLSSVSEDTV...,ENSG00000167528,ENSP00000437832,Gal4 DB fusion and luciferase assay,...,H,Q96N77,157-248,100.0,Q96N77,157-248,100.0,Successful BLAST Hit (>95%),ENSP00000449974,PDPQDLEERDILRVTYTGDGSEHEGDTPELEAEPPRMLSSVSEDTV...


### All Studies Together

In [2]:
import os
import pandas as pd
import numpy as np
import subprocess

# ==========================================
# INPUTS & PATHS
# ==========================================
INPUT_FILE = "/mnt/c/Users/georg/Downloads/UpdatedTable.xlsx"
BLAST_DB = "./blast_table/custom_db/human_proteome_db"
CANONICAL_META_CSV = "./blast_table/TF_completeIDs_with_ensembl_canonical.csv"
OUTPUT_PREFIX = "Master_Effector_DB"

SHEET_CONFIGS = {
    "Tycko2020_NucRep": {"gene": "Gene entry name", "seq": "Extended Domain sequence", "uniprot": None, "coords": None, "ensp": None, "is_nucleotide": False},
    "Tycko2020_NucAct": {"gene": "Gene entry name", "seq": "Extended Domain sequence", "uniprot": None, "coords": None, "ensp": None, "is_nucleotide": False},
    "Tycko2020_TilingRep": {"gene": "Gene ID", "seq": "Sequence", "uniprot": "UniProt ID", "coords": None, "ensp": "Ensembl ID", "is_nucleotide": True},
    "Arnold2018_AD": {"gene": "TF_name", "seq": "tAD sequence [aa]", "uniprot": None, "coords": None, "ensp": None, "is_nucleotide": False},
    "Klaus2023_RD": {"gene": "RD.name", "seq": "RD.region.aaSeq", "uniprot": None, "coords": None, "ensp": None, "is_nucleotide": False},
    "DelRosso_AD": {"gene": "HGNC symbol", "seq": "Sequence", "uniprot": "UniProt ID", "coords": None, "ensp": None, "is_nucleotide": False},
    "DelRosso_RD": {"gene": "HGNC symbol", "seq": "Sequence", "uniprot": "UniProt ID", "coords": None, "ensp": None, "is_nucleotide": False},
    "Tycko2024_AD": {"gene": "HGNC symbol", "seq": "Sequence", "uniprot": "UniProt ID", "coords": None, "ensp": None, "is_nucleotide": False},
    "Tycko2024_RD": {"gene": "HGNC symbol", "seq": "Domain sequence", "uniprot": "UniProt ID", "coords": None, "ensp": None, "is_nucleotide": False},
    "Alerasool2022_AD": {"gene": "Gene", "seq": "Sequence", "uniprot": None, "coords": None, "ensp": None, "is_nucleotide": False}
}

# --- DNA Translator Function ---
def translate_dna(seq):
    if pd.isna(seq): return seq
    codon_table = {
        'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M', 'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
        'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K', 'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',
        'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L', 'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
        'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q', 'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
        'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V', 'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
        'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E', 'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
        'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S', 'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
        'TAC':'Y', 'TAT':'Y', 'TAA':'_', 'TAG':'_', 'TGA':'_', 'TGC':'C', 'TGT':'C', 'TGG':'W',
    }
    seq = str(seq).strip().upper().replace('U', 'T')
    protein = ""
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        if len(codon) == 3:
            aa = codon_table.get(codon, 'X')
            if aa == '_': break
            protein += aa
    return protein

# --- Universal Coordinate 0-to-1 Index Standardizer ---
def standardize_coordinates(row, cfg, sheet_name):
    start, end = None, None
    if sheet_name == "Tycko2020_TilingRep":
        start, end = row.get('start'), row.get('end')
    elif sheet_name in ["DelRosso_AD", "DelRosso_RD", "Tycko2024_AD", "Tycko2024_RD"]:
        start, end = row.get('Start'), row.get('End')
    elif sheet_name in ["Tycko2020_NucRep", "Tycko2020_NucAct"]:
        s, l = row.get('Domain start'), row.get('Domain length')
        if pd.notna(s) and pd.notna(l):
            start = s
            end = int(s) + int(l)
    elif cfg['coords'] and pd.notna(row.get(cfg['coords'])):
        val = str(row[cfg['coords']]).replace('–', '-').strip()
        if '-' in val:
            start, end = val.split('-')[0], val.split('-')[1]
            
    try:
        if pd.notna(start) and pd.notna(end):
            # THE CRITICAL MATH: All sheets get Start + 1
            s_int = int(float(start)) + 1
            e_int = int(float(end))
            return f"{s_int}-{e_int}"
    except:
        pass
    return pd.NA


print(f"\n--- 1. Aggregating Sheets from {INPUT_FILE} ---")
xls = pd.ExcelFile(INPUT_FILE)
all_dfs = []

for sheet_name, cfg in SHEET_CONFIGS.items():
    if sheet_name in xls.sheet_names:
        df_sheet = pd.read_excel(xls, sheet_name=sheet_name)
        df_sheet['Original_Row_Index'] = df_sheet.index + 2 
        
        df_sheet['STD_GENE'] = df_sheet[cfg['gene']] if cfg['gene'] else pd.NA
        df_sheet['STD_SEQ'] = df_sheet[cfg['seq']] if cfg['seq'] else pd.NA
        df_sheet['STD_UNIPROT'] = df_sheet[cfg['uniprot']] if cfg['uniprot'] else pd.NA
        df_sheet['STD_COORDS'] = df_sheet.apply(lambda r: standardize_coordinates(r, cfg, sheet_name), axis=1)
        
        if cfg.get('is_nucleotide'):
            print(f"Translating Nucleotides to Amino Acids for sheet: {sheet_name}")
            df_sheet['STD_SEQ'] = df_sheet['STD_SEQ'].apply(translate_dna)
            
        df_sheet['Provenance'] = sheet_name
        df_sheet = df_sheet.dropna(subset=['STD_SEQ'])
        all_dfs.append(df_sheet)
        print(f"Loaded {len(df_sheet)} valid sequences from {sheet_name}")
    else:
        print(f"Warning: Sheet '{sheet_name}' not found.")

df_master = pd.concat(all_dfs, ignore_index=True)
df_master['original_index'] = df_master.index 
print(f"\nCreated Giant Master Table with {len(df_master)} total domains.")

print("\n--- 2. Generating Master FASTA ---")
fasta_out = f"./blast_table/{OUTPUT_PREFIX}_queries.fasta"
with open(fasta_out, 'w') as f:
    for _, row in df_master.iterrows():
        gene = str(row['STD_GENE']).strip().replace(' ', '_')
        uprot = str(row['STD_UNIPROT']).strip()
        idx = row['original_index']
        f.write(f">query_{idx}_{gene}_{uprot}\n{row['STD_SEQ']}\n")
print(f"FASTA saved to {fasta_out}")

print("\n--- 3. Running BLAST against Canonical DB ---")
blast_out = f"./blast_table/{OUTPUT_PREFIX}_blast_results.tsv"
blast_cmd = [
    "blastp", "-query", fasta_out, "-db", BLAST_DB, "-out", blast_out,
    "-evalue", "1000", "-seg", "no", "-comp_based_stats", "F",
    "-max_target_seqs", "1", "-num_threads", "6",
    "-outfmt", "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore"
]

try:
    subprocess.run(blast_cmd, check=True)
    print("BLAST completed successfully.")
except subprocess.CalledProcessError as e:
    print(f"BLAST execution failed: {e}")
    raise

print("\n--- 4. Loading Canonical Metadata Dictionary ---")
df_meta = pd.read_csv(CANONICAL_META_CSV)
df_meta['Translation ID Clean'] = df_meta['Translation ID'].astype(str).str.split('.').str[0]
df_meta['Gene_Upper'] = df_meta['Gene name (HGNC)'].astype(str).str.strip().str.upper()
df_meta['UniProt_Clean'] = df_meta['UniProt_ID'].astype(str).str.split('-').str[0] 

def build_canonical_dict(group_col):
    mapping = {}
    for key, group in df_meta.groupby(group_col):
        if key == 'NAN' or pd.isna(key) or key == '': continue
        if len(group) == 1:
            mapping[key] = group['Translation ID Clean'].iloc[0]
        else:
            canon_rows = group[group['Ensembl canonical'].astype(str).str.strip().str.lower() == 'canonical']
            if not canon_rows.empty:
                mapping[key] = canon_rows['Translation ID Clean'].iloc[0]
            else:
                mapping[key] = group['Translation ID Clean'].iloc[0]
    return mapping

gene_to_ensp = build_canonical_dict('Gene_Upper')
uniprot_to_ensp = build_canonical_dict('UniProt_Clean')
ensp_to_seq = dict(zip(df_meta['Translation ID Clean'], df_meta['Sequence aa']))

print("\n--- 5. Parsing BLAST & Applying Pipeline Logic ---")
blast_cols = ['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']

if os.path.exists(blast_out) and os.path.getsize(blast_out) > 0:
    df_blast = pd.read_csv(blast_out, sep='\t', names=blast_cols)
    df_blast['original_index'] = df_blast['qseqid'].apply(lambda x: int(x.split('_')[1]))
    df_blast = df_blast.drop_duplicates(subset=['original_index'], keep='first')
    
    def extract_uniprot_id(s):
        if pd.isna(s): return s
        parts = str(s).split('|')
        if len(parts) >= 2: return parts[1]
        return s
    
    df_blast['BLAST_UNIPROT_ID'] = df_blast['sseqid'].apply(extract_uniprot_id)
    df_blast['BLAST_DomainCoordinates'] = df_blast['sstart'].astype(str) + '-' + df_blast['send'].astype(str)
    df_blast = df_blast.rename(columns={'pident': 'BLAST_Identity_Pct'})
    
    merge_cols = ['original_index', 'BLAST_UNIPROT_ID', 'BLAST_DomainCoordinates', 'BLAST_Identity_Pct']
    df_blast_meta = df_blast[merge_cols].set_index('original_index')
    df_final = df_master.join(df_blast_meta, on='original_index')

    def determine_final_humanization(row):
        ident = float(row['BLAST_Identity_Pct']) if pd.notna(row['BLAST_Identity_Pct']) else 0.0
        if ident > 95.0:
            return row['BLAST_UNIPROT_ID'], row['BLAST_DomainCoordinates'], ident, "Successful BLAST Hit (>95%)"
        else:
            reason = f"Suspicious Hit ({ident}%) - Requires Strict Verification" if ident > 0 else "No BLAST Hit - Requires Strict Verification"
            return row['STD_UNIPROT'], row['STD_COORDS'], ident, reason

    df_final[['Final_Human_UNIPROT_ID', 'Final_Human_DomainCoords', 'Final_Identity', 'Mapping_Status']] = df_final.apply(
        determine_final_humanization, axis=1, result_type='expand'
    )

    def get_final_ensp(row):
        if 'Requires Strict Verification' in str(row.get('Mapping_Status', '')):
            return pd.NA 
            
        if pd.notna(row['Final_Human_UNIPROT_ID']) and str(row['Final_Human_UNIPROT_ID']).lower() != 'nan':
            uprot = str(row['Final_Human_UNIPROT_ID']).split('-')[0]
            if uprot in uniprot_to_ensp: return uniprot_to_ensp[uprot]

        gene = str(row['STD_GENE']).strip().upper()
        if gene in gene_to_ensp: return gene_to_ensp[gene]
        return pd.NA

    df_final['Final_Human_ENSP_ID'] = df_final.apply(get_final_ensp, axis=1)

    print("\n--- 6. The Strict Bouncer: Verifying Non-Canonical Sequences ---")
    def rescue_missing_ensp_and_coords(row):
        ensp = row['Final_Human_ENSP_ID']
        status = str(row['Mapping_Status'])
        coords = row['Final_Human_DomainCoords']
        
        if 'Requires Strict Verification' not in status and pd.notna(ensp): 
            return ensp, status, coords
            
        gene = str(row['STD_GENE']).strip().upper()
        gene_clean = gene.split('_')[0] if '_' in gene else gene
        gene_rows = df_meta[df_meta['Gene_Upper'].isin([gene, gene_clean])].copy()
        
        if gene_rows.empty: 
            return pd.NA, "Failed: Gene not in Human DB (Non-Human)", pd.NA
            
        target_seq = str(row['STD_SEQ']).strip()
        orig_coords = str(row['STD_COORDS']).strip()
        
        # --- THE LENGTH TRAP FIX ---
        req_length = 0
        if orig_coords and '-' in orig_coords:
            try:
                req_length = int(orig_coords.split('-')[1])
            except: pass
                
        gene_rows['Protein length'] = pd.to_numeric(gene_rows['Protein length'], errors='coerce').fillna(0)
        
        # Strictly filter out any isoforms physically shorter than the required End coordinate!
        valid_length_rows = gene_rows[gene_rows['Protein length'] >= req_length]
        search_rows = valid_length_rows if not valid_length_rows.empty else gene_rows
        
        # Sort by length descending, so fallbacks always grab the longest safe option
        search_rows = search_rows.sort_values(by='Protein length', ascending=False)
        # ---------------------------
        
        if target_seq and target_seq != 'nan':
            for _, g_row in search_rows.iterrows():
                full_seq = str(g_row['Sequence aa'])
                if target_seq in full_seq:
                    start_idx = full_seq.find(target_seq) + 1
                    end_idx = start_idx + len(target_seq) - 1
                    new_coords = f"{start_idx}-{end_idx}"
                    
                    is_canon = (str(g_row['Ensembl canonical']).strip().lower() == 'canonical')
                    if orig_coords == new_coords:
                        return g_row['Translation ID Clean'], "Confirmed Human (Seq + Exact Coord Match)", new_coords
                    if is_canon:
                        return g_row['Translation ID Clean'], "Confirmed Human (Seq Match -> Canonical)", new_coords
                    return g_row['Translation ID Clean'], "Confirmed Human (Seq Match -> Longest Isoform)", new_coords

        target_uprot = str(row['STD_UNIPROT']).strip()
        if target_uprot and target_uprot != 'nan':
            for _, g_row in search_rows.iterrows():
                if str(g_row['UniProt_ID']).strip() == target_uprot:
                    return g_row['Translation ID Clean'], "Confirmed Human (UniProt Provided by Paper)", orig_coords
            
        # If sequence and UniProt match fail, safely fall back to Canonical or Longest (Pre-filtered by length!)
        for _, g_row in search_rows.iterrows():
            if str(g_row['Ensembl canonical']).strip().lower() == 'canonical':
                return g_row['Translation ID Clean'], status + " (Rescued Canonical by Gene)", orig_coords
                
        longest_row = search_rows.iloc[0]
        return longest_row['Translation ID Clean'], status + " (Rescued Longest Isoform by Gene)", orig_coords

    df_final[['Final_Human_ENSP_ID', 'Mapping_Status', 'Final_Human_DomainCoords']] = df_final.apply(
        rescue_missing_ensp_and_coords, axis=1, result_type='expand'
    )

    print("\n--- 7. Extracting Final Sequences ---")
    def get_final_sequence(row):
        ensp = str(row.get('Final_Human_ENSP_ID', ''))
        coords = str(row.get('Final_Human_DomainCoords', ''))
        
        if ensp == 'nan' or ensp == '<NA>' or not coords or coords == 'nan': 
            return pd.NA
            
        full_seq = str(ensp_to_seq.get(ensp, ""))
        if not full_seq or full_seq == 'nan': 
            return pd.NA
            
        coords = coords.replace('–', '-').replace('—', '-')
        try:
            if '-' in coords:
                start_str, end_str = coords.split('-')
                start = int(start_str.strip()) - 1
                end = int(end_str.strip())
                return full_seq[start:end]
        except Exception: pass
        return pd.NA

    df_final['Final_Human_Domain_Sequence'] = df_final.apply(get_final_sequence, axis=1)

    print("\n--- 8. Harmonization, Column Cleanup & Save ---")
    redundant_raw_cols = [
        "Gene entry name", "TF_name", "RD.name", "HGNC symbol", "Gene", "Gene ID",
        "Domain start", "start", "Start", "Coordinates",
        "Domain length", "end", "End", "RD.region.length.AA", "tAD length [aa]",
        "Pfam Domain sequence", "Extended Domain sequence", "Sequence", "tAD sequence [aa]", "RD.region.aaSeq", "Domain sequence",
        "Ensembl ID", "Transcript ID", "UniProt ID", "Uniprot ID",
        "Fragment", "logFC high GFP", "FDR high GFP", "Hit high GFP", "logFC medium GFP", "FDR medium GFP", "Hit medium GFP"
    ]
    df_final = df_final.drop(columns=[c for c in redundant_raw_cols if c in df_final.columns], errors='ignore')

    # Remove original_index from rename_dict!
    rename_dict = {
        "STD_GENE": "Original_Gene",
        "STD_SEQ": "Original_Sequence",
        "STD_UNIPROT": "Original_UniProt_ID"
    }
    df_final = df_final.rename(columns=rename_dict)
    
    cols_to_front = [
        'Original_Row_Index', 'Provenance', 'Original_Gene', 'Mapping_Status', 'Final_Human_ENSP_ID', 
        'Final_Human_DomainCoords', 'Final_Human_Domain_Sequence', 'Original_Sequence', 'Original_UniProt_ID'
    ]
    
    mapping_metrics = [
        'Final_Identity', 'Final_Human_UNIPROT_ID',
        'BLAST_UNIPROT_ID', 'BLAST_Identity_Pct'
    ]
    
    # Explicitly exclude the internal 'original_index' from the final columns
    other_cols = [
        c for c in df_final.columns 
        if c not in cols_to_front 
        and c not in mapping_metrics 
        and not c.startswith('STD_')
        and c != 'original_index'
    ]
    
    final_col_order = cols_to_front + other_cols + mapping_metrics
    df_final = df_final[[c for c in final_col_order if c in df_final.columns]]

    final_out = f"{OUTPUT_PREFIX}_processed.xlsx"
    df_final.to_excel(final_out, index=False)
    print(f"Saved harmonized table to: '{final_out}'")
    
    missing_df = df_final[df_final['Final_Human_ENSP_ID'].isna()]
    if not missing_df.empty:
        missing_out = f"{OUTPUT_PREFIX}_missing_IDs.xlsx"
        missing_df.to_excel(missing_out, index=False)
        print(f"Saved {len(missing_df)} domains missing Ensembl IDs (Rejected Orthologs) to: '{missing_out}'")

    print(f"\nPIPELINE COMPLETE.")

else:
    print(f"\nNo BLAST results found in {blast_out}. Check your FASTA or DB.")


--- 1. Aggregating Sheets from /mnt/c/Users/georg/Downloads/UpdatedTable.xlsx ---
Loaded 446 valid sequences from Tycko2020_NucRep
Loaded 48 valid sequences from Tycko2020_NucAct
Translating Nucleotides to Amino Acids for sheet: Tycko2020_TilingRep
Loaded 385 valid sequences from Tycko2020_TilingRep
Loaded 53 valid sequences from Arnold2018_AD
Loaded 195 valid sequences from Klaus2023_RD
Loaded 374 valid sequences from DelRosso_AD
Loaded 715 valid sequences from DelRosso_RD
Loaded 38 valid sequences from Tycko2024_AD
Loaded 1223 valid sequences from Tycko2024_RD
Loaded 71 valid sequences from Alerasool2022_AD

Created Giant Master Table with 3548 total domains.

--- 2. Generating Master FASTA ---
FASTA saved to ./blast_table/Master_Effector_DB_queries.fasta

--- 3. Running BLAST against Canonical DB ---


BLAST completed successfully.

--- 4. Loading Canonical Metadata Dictionary ---

--- 5. Parsing BLAST & Applying Pipeline Logic ---

--- 6. The Strict Bouncer: Verifying Non-Canonical Sequences ---

--- 7. Extracting Final Sequences ---

--- 8. Harmonization, Column Cleanup & Save ---
Saved harmonized table to: 'Master_Effector_DB_processed.xlsx'
Saved 1237 domains missing Ensembl IDs (Rejected Orthologs) to: 'Master_Effector_DB_missing_IDs.xlsx'

PIPELINE COMPLETE.


In [4]:
import pandas as pd
import numpy as np

print("--- 1. Loading Databases ---")

# Load Table 1 (TFRegDB1)
df1 = pd.read_excel("/mnt/c/Users/georg/Downloads/TFRegDB1_Humanized.xlsx")
df1 = df1.rename(columns={'TF name': 'Original_Gene'})
df1['Source_Database'] = 'TFRegDB1'
df1['Original_Row_Index'] = df1.index + 2 

# Load Table 2 (HumanizedTable)
df2 = pd.read_excel("/mnt/c/Users/georg/Downloads/HumanizedTable.xlsx")
df2 = df2.rename(columns={'Gene': 'Original_Gene'})
df2['Source_Database'] = 'HumanizedTable'
if 'original_row' in df2.columns:
    df2['Original_Row_Index'] = df2['original_row']
else:
    df2['Original_Row_Index'] = df2.index + 2

# Load Table 3 (Master Effector DB)
df3 = pd.read_excel("Master_Effector_DB_processed.xlsx")
df3['Source_Database'] = df3['Provenance']
# Original_Row_Index already exists in this table from our master pipeline

print("--- 2. Merging and Filtering Data ---")
df_master = pd.concat([df1, df2, df3], ignore_index=True)

# Clean up row index decimals (e.g., 42.0 -> 42)
df_master['Original_Row_Index'] = df_master['Original_Row_Index'].fillna(0).astype(str).str.replace('.0', '', regex=False)

# Filter explicitly for rows that have a valid Final_Human_ENSP_ID
df_valid = df_master.dropna(subset=['Final_Human_ENSP_ID']).copy()
invalid_strings = ['nan', '<na>', 'none', '', 'na']
df_valid = df_valid[~df_valid['Final_Human_ENSP_ID'].astype(str).str.strip().str.lower().isin(invalid_strings)]

print(f"Total domains loaded: {len(df_master)}")
print(f"Successfully humanized domains kept: {len(df_valid)}")


print("--- 3. Generating Protein BED Coordinates (1-Indexed) ---")
def parse_bed_coords(row):
    coords = str(row.get('Final_Human_DomainCoords', '')).strip()
    if '-' in coords:
        try:
            start_str, end_str = coords.split('-')
            # STRICT 1-INDEXED MATH (No longer subtracting 1)
            start = int(start_str.strip())
            end = int(end_str.strip())
            return pd.Series([start, end])
        except Exception:
            return pd.Series([pd.NA, pd.NA])
    return pd.Series([pd.NA, pd.NA])

df_valid[['BED_Start', 'BED_End']] = df_valid.apply(parse_bed_coords, axis=1)
df_bed_ready = df_valid.dropna(subset=['BED_Start', 'BED_End']).copy()
df_bed_ready['BED_Start'] = df_bed_ready['BED_Start'].astype(int)
df_bed_ready['BED_End'] = df_bed_ready['BED_End'].astype(int)


print("--- 4. Exporting Files ---")
bed_df = pd.DataFrame({
    'chrom': df_bed_ready['Final_Human_ENSP_ID'],
    'chromStart': df_bed_ready['BED_Start'],
    'chromEnd': df_bed_ready['BED_End'],
    # Combine Gene, Source, AND Row Number for ultimate tracking
    'name': df_bed_ready['Original_Gene'].astype(str) + "_" + \
            df_bed_ready['Source_Database'].astype(str) + "_Row" + \
            df_bed_ready['Original_Row_Index'].astype(str),
    'score': df_bed_ready['Final_Identity'].fillna(0).astype(str),
    'strand': '.' 
})

bed_df = bed_df.sort_values(by=['chrom', 'chromStart'])

bed_out = "Unified_Effector_Domains.bed"
bed_df.to_csv(bed_out, sep='\t', header=False, index=False)

excel_out = "Unified_Valid_Domains.xlsx"
df_valid_export = df_valid.drop(columns=['BED_Start', 'BED_End'], errors='ignore')
df_valid_export.to_excel(excel_out, index=False)

print(f"Process complete.")
print(f"- Saved Master Unified Dataframe to: '{excel_out}'")
print(f"- Saved {len(bed_df)} domains to 1-Indexed BED file: '{bed_out}'")

# --- Quick Diagnostic for the 6 Missing Domains ---
# Find the rows in df_valid that failed coordinate parsing
df_missing_coords = df_valid[df_valid['BED_Start'].isna() | df_valid['BED_End'].isna()]

print(f"\n--- WARNING: {len(df_missing_coords)} domains failed BED coordinate parsing ---")
if not df_missing_coords.empty:
    cols_to_show = ['Original_Gene', 'Source_Database', 'Original_Row_Index', 'Final_Human_ENSP_ID', 'Final_Human_DomainCoords']
    print(df_missing_coords[cols_to_show])
    
    # Optional: Save them to a separate file for easy fixing
    df_missing_coords.drop(columns=['BED_Start', 'BED_End'], errors='ignore').to_excel("Broken_Coordinates_Warning.xlsx", index=False)
    print("Saved these specific domains to 'Broken_Coordinates_Warning.xlsx' for manual review.")

--- 1. Loading Databases ---
--- 2. Merging and Filtering Data ---
Total domains loaded: 5525
Successfully humanized domains kept: 4069
--- 3. Generating Protein BED Coordinates (1-Indexed) ---
--- 4. Exporting Files ---
Process complete.
- Saved Master Unified Dataframe to: 'Unified_Valid_Domains.xlsx'
- Saved 4063 domains to 1-Indexed BED file: 'Unified_Effector_Domains.bed'

--- WARNING: 6 domains failed BED coordinate parsing ---
        Original_Gene Source_Database Original_Row_Index Final_Human_ENSP_ID  \
2896  otp_FBtr0110772   Arnold2018_AD                 42     ENSP00000302814   
3004            FoxL1    Klaus2023_RD                 97     ENSP00000326272   
3011              Gsc    Klaus2023_RD                104     ENSP00000238558   
3041              Mnt    Klaus2023_RD                134     ENSP00000174618   
3062           Prdm13    Klaus2023_RD                155     ENSP00000358217   
3079              Sp1    Klaus2023_RD                172     ENSP00000329357   

 

In [2]:
# read .bed, get ENSP id and count unique ENSP ids
import pandas as pd
bed_df = pd.read_csv("Unified_Effector_Domains.bed", sep='\t', header=None, names=['chrom', 'chromStart', 'chromEnd', 'name', 'score', 'strand'])
unique_ensp_count = bed_df['chrom'].nunique()
print(f"\nTotal unique ENSP IDs in BED file: {unique_ensp_count}")


Total unique ENSP IDs in BED file: 1293
